# Train GONet (MSD) on Kaggle

1. Attach the labeled Kaggle inputs for `HYDR`, `DRISHTI-GS`, and `PAPILA`. The notebook creates `/kaggle/working/gonet-data/` aliases from those separate inputs.
2. **REFUGE is not required.** `REFUGE2` is not used as a substitute because it has no glaucoma labels.
3. Set Notebook Settings to **GPU** and enable **Internet** so packages and pretrained DINOv2 weights can be fetched.
4. Run all cells. With these three labeled domains, the default runs three leave-one-domain-out folds; set `TARGET_DOMAIN` to one domain for a single fold while iterating.

Source validation is split by patient ID. The notebook writes per-image held-out probabilities, aggregate metrics, and checkpoints under `/kaggle/working/gonet_msd_runs/`. Push the updated repository code to GitHub `main` before running, because the first cell clones or pulls that branch.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/TaiTranDang145/Glaucoma-Detection.git"
REPO_DIR = Path("/kaggle/working/Glaucoma-Detection")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)

required_files = [
    REPO_DIR / "src/train_msd.py",
    REPO_DIR / "src/models/gonet.py",
    REPO_DIR / "configs/gonet_msd.yaml",
]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise RuntimeError("GONet MSD code is not on GitHub main yet. Push the local implementation, then rerun this cell. Missing: " + ", ".join(missing))

print(f"Source ready at {REPO_DIR}")


In [ ]:
%cd /kaggle/working/Glaucoma-Detection
%pip install -q timm omegaconf tensorboard scikit-learn "albumentations<2"


In [ ]:
from pathlib import Path
import torch

KAGGLE_OWNER_ROOT = Path("/kaggle/input/datasets/taitrandang")
DATA_ROOT = Path("/kaggle/working/gonet-data")
if not KAGGLE_OWNER_ROOT.is_dir():
    raise FileNotFoundError(f"Kaggle input root not found: {KAGGLE_OWNER_ROOT}; inspect /kaggle/input and update the path.")
mounts = [path for path in KAGGLE_OWNER_ROOT.iterdir() if path.is_dir()]
print("Attached dataset folders:", [path.name for path in mounts])

def find_mount(token):
    found = [path for path in mounts if token in path.name.casefold()]
    if len(found) != 1:
        raise RuntimeError(f"Expected one attached dataset matching {token!r}; found: {found}")
    return found[0]

def replace_link(alias, target):
    link = DATA_ROOT / alias
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise FileExistsError(f"Refusing to replace existing non-link path: {link}")
    link.symlink_to(target, target_is_directory=True)

DATA_ROOT.mkdir(parents=True, exist_ok=True)
hydr_mount = find_mount("hydr")
hydr_roots = [p.parent for p in hydr_mount.rglob("Labels.csv") if (p.parent / "Images").is_dir()]
if len(hydr_roots) != 1:
    raise RuntimeError(f"HYDR needs one Labels.csv with a sibling Images/ directory; found: {hydr_roots}")
replace_link("HYDR", hydr_roots[0])
replace_link("DRISHTI-GS", find_mount("drishti"))
replace_link("PAPILA", find_mount("papila"))

for required in ("HYDR/Labels.csv", "DRISHTI-GS", "PAPILA"):
    assert (DATA_ROOT / required).exists(), f"Missing {DATA_ROOT / required}; check the attached Kaggle inputs."
papila_root = DATA_ROOT / "PAPILA"
if not any(p.is_dir() for p in papila_root.rglob("FundusImages")) or not any(p.is_dir() for p in papila_root.rglob("ClinicalData")):
    raise RuntimeError("PAPILA input must contain FundusImages/ and ClinicalData/.")

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Dataset root:", DATA_ROOT)
print("Labeled training domains: HYDR, DRISHTI_GS, PAPILA. REFUGE2 is not used.")


In [ ]:
from pathlib import Path
import subprocess
import sys

MANIFEST = Path("/kaggle/working/gonet_msd.csv")
OUTPUT_DIR = Path("/kaggle/working/gonet_msd_runs")
EPOCHS = 20
BATCH_SIZE = 16  # Paper setting; lower only if GPU memory is insufficient.
TARGET_DOMAIN = None  # Example: "PAPILA" to run only one held-out domain.

subprocess.run([
    sys.executable, "src/prepare_data.py",
    "--data-root", str(DATA_ROOT),
    "--output", str(MANIFEST),
], check=True)

command = [
    sys.executable, "src/train_msd.py",
    "--config", "configs/gonet_msd.yaml",
    "--data-root", str(DATA_ROOT),
    "--manifest", str(MANIFEST),
    "--output-dir", str(OUTPUT_DIR),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", "2",
]
if TARGET_DOMAIN:
    command.extend(["--target-domain", TARGET_DOMAIN])
subprocess.run(command, check=True)


In [ ]:
from pathlib import Path

for path in sorted(Path("/kaggle/working/gonet_msd_runs").rglob("*")):
    if path.is_file():
        print(path, f"({path.stat().st_size / 1024 / 1024:.1f} MB)")

print("Combined held-out probabilities:", Path("/kaggle/working/gonet_msd_runs/gonet_predictions.csv"))
